# Test PaddleOCR-VL-1.6

So sánh nhanh với PaddleOCR classic (`2_OCR-Paddle.ipynb`) trên cùng 1 file. Chỉ sửa `SOURCE_PATH` và `PAGE_NUMBER` bên dưới rồi chạy tuần tự.

### Cài đặt (chạy 1 lần)
`paddleocr` bản >= 3.x mới có class `PaddleOCRVL`. Nếu đã cài `paddleocr` cho notebook classic rồi thì chỉ cần `pip install -U paddleocr`.

In [2]:
import sys
print("Kernel Python:", sys.executable)

Kernel Python: c:\2_Company\GSOFT\Enterprice-Chatbot\BVBank-Chatbot\dev_llm_service\.venv\Scripts\python.exe


In [1]:
%pip install -U paddleocr paddlepaddle pymupdf

Note: you may need to restart the kernel to use updated packages.


c:\2_Company\GSOFT\Enterprice-Chatbot\BVBank-Chatbot\dev_llm_service\.venv\Scripts\python.exe: No module named pip


In [3]:
import time
from pathlib import Path

import fitz  # PyMuPDF
from paddleocr import PaddleOCRVL

ModuleNotFoundError: No module named 'fitz'

### Cấu hình input
`SOURCE_PATH` có thể là ảnh (`.png/.jpg`) hoặc PDF. Nếu là PDF thì cell render bên dưới sẽ cắt `PAGE_NUMBER` ra thành ảnh trước khi OCR.

In [5]:
SOURCE_PATH = Path("../my_documents/Bai_Tap_SQL_to_Text_Agent.pdf")
PAGE_NUMBER = 1  # chỉ dùng nếu SOURCE_PATH là PDF
OUTPUT_DIR = Path("output_paddleocr_vl")
OUTPUT_DIR.mkdir(exist_ok=True)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {SOURCE_PATH}")

print("Input file:", SOURCE_PATH)

Input file: ..\my_documents\Bai_Tap_SQL_to_Text_Agent.pdf


In [6]:
def resolve_image_path(source_path: Path, page_number: int) -> Path:
    """Trả về đường dẫn ảnh để OCR. Nếu source là PDF thì render trang ra PNG."""
    if source_path.suffix.lower() != ".pdf":
        return source_path

    doc = fitz.open(str(source_path))
    page = doc[page_number - 1]
    pix = page.get_pixmap(dpi=400, alpha=False)

    image_path = OUTPUT_DIR / f"{source_path.stem}_page_{page_number}.png"
    pix.save(str(image_path))
    doc.close()
    return image_path


image_path = resolve_image_path(SOURCE_PATH, PAGE_NUMBER)
print("Ảnh dùng để OCR:", image_path)

NameError: name 'fitz' is not defined

### Khởi tạo PaddleOCR-VL và chạy OCR

In [ ]:
pipeline = PaddleOCRVL()

In [ ]:
start = time.perf_counter()
output = pipeline.predict(str(image_path))
elapsed = time.perf_counter() - start

print("=" * 100)
print(f"PaddleOCR-VL time: {elapsed:.2f}s")
print("=" * 100)

for res in output:
    if res is None:
        continue
    res.print()
    res.save_to_json(save_path=str(OUTPUT_DIR))
    res.save_to_markdown(save_path=str(OUTPUT_DIR))

print(f"\nĐã lưu kết quả vào: {OUTPUT_DIR.resolve()}")

### (Tuỳ chọn) So sánh với PaddleOCR classic (PP-OCRv6)
Chạy cell dưới để đối chiếu trực tiếp với bản đang dùng trong `2_OCR-Paddle.ipynb`.

In [ ]:
from paddleocr import PaddleOCR

classic_ocr = PaddleOCR(use_textline_orientation=True, lang="vi")

start = time.perf_counter()
classic_result = classic_ocr.predict(str(image_path))
elapsed = time.perf_counter() - start

print("=" * 100)
print(f"PaddleOCR classic (PP-OCRv6) time: {elapsed:.2f}s")
print("=" * 100)

for page_result in classic_result:
    if page_result is None:
        continue
    for text, score in zip(
        page_result.get("rec_texts", []), page_result.get("rec_scores", [])
    ):
        print(f"{score:<8.3f} {text}")